# Road damage detection: train Faster R-CNN (ResNet-50 FPN) in the cloud

Trains Faster R-CNN (ResNet-50 FPN) on RDD2022 (classes D00, D10, D20, D40), then evaluates it on the val
and test splits. The notebook is self-contained: it writes the project scripts itself,
so you don't need to clone the repository. It was generated by
`scripts/build_cloud_notebooks.py`; edit the scripts and rebuild instead of editing
the code cells by hand.

**Kaggle**
1. *Create → New Notebook → File → Import Notebook*, then upload this file.
2. *Add Input*: the RDD2022 YOLO dataset (`train/ val/ test/` with `images/` and `labels/`),
   or your own fixed `data/` folder uploaded as a private dataset.
3. *Settings*: Accelerator **GPU T4 x2** or **P100**, Internet **on**.
4. *Save Version → Save & Run All (Commit)*. This runs for up to 12 h and keeps `/kaggle/working`.
5. If training stopped at `MAX_HOURS`, create a new version, *Add Input → Your Work →* the
   previous version's output, set `RESUME_FROM` to that path, and commit again. Training
   continues from the last epoch.

**Colab**
1. Upload the notebook, then *Runtime → Change runtime type → GPU*.
2. Put the dataset (folder or `.zip`) on Google Drive and set `DATA_DIR`.
3. Set `DRIVE_DIR` so runs are saved to Drive. After a disconnect, *Run all* again and
   training resumes automatically.

The data is fixed on the fly by `fix_prepared_data.py`. Class 4 (pothole) is remapped
to D40, "other corruption" boxes are dropped, and images are split by 50-frame blocks
so neighbouring frames don't leak between splits. Data that was already fixed is used as-is.

## 1. Settings

In [ ]:
# ---- settings: edit these ----
MODEL_KEY = "faster_rcnn"     # run name prefix; runs are named <MODEL_KEY>_<IMGSZ>_s<SEED>
EPOCHS = 30               # keep the same for every model you compare
IMGSZ = 640
BATCH = 8               # for a 16 GB GPU; lower it if CUDA runs out of memory
SEED = 42
DEVICE = "0"              # "cpu" to test without a GPU
MAX_HOURS = 10.5          # stop cleanly before the session limit, then run again to resume
DATA_DIR = None           # folder with train/ val/ test/ (or a .zip of it); None = search inputs
DRIVE_DIR = None          # Colab: e.g. "/content/drive/MyDrive/road-damage" keeps runs on Drive
RESUME_FROM = None        # Kaggle: previous version's output added as input, to continue training
EXTRA_ARGS = ""           # extra training flags, e.g. "--fraction 0.05" for a quick test
AMP = True                # mixed precision on the GPU (set False for GTX 16xx cards)

## 2. Environment

In [ ]:
import glob, json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB and (DRIVE_DIR or str(DATA_DIR or "").startswith("/content/drive")):
    from google.colab import drive
    drive.mount("/content/drive")

if ON_KAGGLE:   # code + data in /tmp, results in /kaggle/working (saved with the version)
    WORK, OUT = Path("/tmp/rdd"), Path("/kaggle/working")
elif ON_COLAB:
    WORK, OUT = Path("/content/rdd"), Path(DRIVE_DIR or "/content/outputs")
else:
    WORK = Path.cwd() / "cloud_work"
    OUT = Path(DRIVE_DIR) if DRIVE_DIR else WORK / "outputs"
RUNS = OUT / "runs"
for d in (WORK / "scripts", WORK / "configs", RUNS):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
RUN_NAME = f"{MODEL_KEY}_{IMGSZ}_s{SEED}"
WORKERS = min(8, os.cpu_count() or 2)
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} | code+data: {WORK} | outputs: {OUT} | run: {RUN_NAME}")


def run(cmd):
    """Run a command, streaming its output into the notebook; raise if it fails."""
    print("$", " ".join(shlex.quote(str(c)) for c in cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    while chunk := proc.stdout.read1(4096):
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    if proc.wait():
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")

In [ ]:
%pip install -q pyyaml tqdm pandas matplotlib

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.0f} GB")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Kaggle: Settings → Accelerator → GPU. Colab: Runtime → Change runtime type → GPU.")

## 3. Project code

These cells are copies of the repository's `scripts/` and `configs/` files.

In [ ]:
%%writefile scripts/utils.py
"""Shared helpers for the road damage detection scripts.

Not meant to be run directly. The other scripts in this folder import from it.
"""
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[1]
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
DEFAULT_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)


# --------------------------------------------------------------------------- #
# General helpers
# --------------------------------------------------------------------------- #
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def save_json(obj, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=float)


def resolve_weights(weights) -> str:
    """Use the path as given, or relative to the project root. Otherwise return it
    unchanged (so names like 'yolov8s.pt' can be auto-downloaded by Ultralytics)."""
    p = Path(weights)
    if p.exists():
        return str(p)
    if (PROJECT_ROOT / p).exists():
        return str(PROJECT_ROOT / p)
    return str(weights)


def _is_rtdetr(weights: str) -> bool:
    if "rtdetr" in weights.lower() or "rt-detr" in weights.lower():
        return True
    p = Path(weights)
    if p.suffix == ".pt" and p.exists():
        try:
            import torch

            ckpt = torch.load(str(p), map_location="cpu", weights_only=False)
            m = ckpt.get("ema") or ckpt.get("model")
            return type(m).__name__ == "RTDETRDetectionModel"
        except Exception:
            return False
    return False


def load_model(weights, task: str | None = "detect"):
    """Load a YOLO or RT-DETR model. RT-DETR needs its own class because it uses a
    different predictor, so it is detected from the file name or the checkpoint."""
    from ultralytics import RTDETR, YOLO

    weights = resolve_weights(weights)
    if _is_rtdetr(weights):
        return RTDETR(weights)
    return YOLO(weights, task=task) if task else YOLO(weights)


# --------------------------------------------------------------------------- #
# Dataset yaml helpers
# --------------------------------------------------------------------------- #
def resolve_data_yaml(data) -> Path:
    """Write a copy of the dataset yaml with an absolute `path:`.

    Ultralytics resolves relative dataset paths against its own datasets directory,
    not the current folder, so we always hand it an absolute path.
    """
    data_path = Path(data)
    if not data_path.exists():
        data_path = PROJECT_ROOT / data
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset yaml not found: {data}")
    with open(data_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    root = Path(cfg.get("path", data_path.parent))
    if not root.is_absolute():
        root = (PROJECT_ROOT / root).resolve()
    cfg["path"] = root.as_posix()
    out_dir = PROJECT_ROOT / "runs" / "_resolved"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / data_path.name
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out


def load_names(data_yaml) -> dict[int, str]:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    names = cfg["names"]
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    return {int(k): v for k, v in names.items()}


def get_split_dirs(data_yaml, split: str) -> tuple[Path, Path]:
    """Return (images_dir, labels_dir) for a split of a resolved dataset yaml."""
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if split not in cfg:
        raise KeyError(f"Split '{split}' is not defined in {data_yaml}")
    rel = cfg[split]
    if isinstance(rel, list):
        rel = rel[0]
    img_dir = Path(cfg["path"]) / rel
    parts = list(img_dir.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            break
    else:
        raise ValueError(f"Cannot derive the labels folder from {img_dir}")
    return img_dir, Path(*parts)


def list_images(folder) -> list[Path]:
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)


def read_yolo_labels(path) -> tuple[np.ndarray, np.ndarray]:
    """Read a YOLO label file. Returns (class_ids[n], boxes_xyxy_normalised[n, 4])."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros(0, dtype=int), np.zeros((0, 4))
    arr = np.loadtxt(path, ndmin=2)
    cls = arr[:, 0].astype(int)
    cx, cy, w, h = arr[:, 1], arr[:, 2], arr[:, 3], arr[:, 4]
    xyxy = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)
    return cls, xyxy


# --------------------------------------------------------------------------- #
# Detection metrics (F1 at IoU 0.5, the RDD challenge metric)
# --------------------------------------------------------------------------- #
def box_iou_np(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pairwise IoU between boxes a[n, 4] and b[m, 4] in xyxy format."""
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.maximum(union, 1e-12)


class DetectionEvaluator:
    """Accumulates predictions over a dataset and computes P / R / F1 at any
    confidence threshold.

    Predictions are matched to ground truth once (greedy, by descending confidence,
    same class, IoU >= iou_thr). Because greedy matching only depends on higher
    scoring boxes, the result for any threshold is just a filter on that single
    matching, so sweeping thresholds is cheap.

    IoU is computed on normalised coordinates, which gives the same value as pixel
    coordinates (IoU is invariant to scaling each axis).
    """

    def __init__(self, nc: int, iou_thr: float = 0.5):
        self.nc = nc
        self.iou_thr = iou_thr
        self.n_gt = np.zeros(nc, dtype=int)
        self._scores, self._cls, self._tp = [], [], []
        self._cache = None

    def add(self, pred_boxes, pred_scores, pred_cls, gt_boxes, gt_cls) -> None:
        self._cache = None
        pred_boxes = np.asarray(pred_boxes, dtype=float).reshape(-1, 4)
        pred_scores = np.asarray(pred_scores, dtype=float).reshape(-1)
        pred_cls = np.asarray(pred_cls).astype(int).reshape(-1)
        gt_boxes = np.asarray(gt_boxes, dtype=float).reshape(-1, 4)
        gt_cls = np.asarray(gt_cls).astype(int).reshape(-1)

        for c in range(self.nc):
            g = gt_boxes[gt_cls == c]
            self.n_gt[c] += len(g)
            m = pred_cls == c
            if not m.any():
                continue
            b, s = pred_boxes[m], pred_scores[m]
            order = np.argsort(-s)
            b, s = b[order], s[order]
            tp = np.zeros(len(b), dtype=bool)
            if len(g):
                ious = box_iou_np(b, g)
                matched = np.zeros(len(g), dtype=bool)
                for i in range(len(b)):
                    cand = np.where(matched, -1.0, ious[i])
                    j = int(cand.argmax())
                    if cand[j] >= self.iou_thr:
                        matched[j] = True
                        tp[i] = True
                        if matched.all():
                            break
            self._scores.append(s)
            self._cls.append(np.full(len(b), c, dtype=int))
            self._tp.append(tp)

    def _arrays(self):
        if self._cache is None:
            if self._scores:
                self._cache = (
                    np.concatenate(self._scores),
                    np.concatenate(self._cls),
                    np.concatenate(self._tp),
                )
            else:
                self._cache = (np.zeros(0), np.zeros(0, dtype=int), np.zeros(0, dtype=bool))
        return self._cache

    @staticmethod
    def _prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f1

    def at_threshold(self, conf: float) -> dict:
        scores, cls, tp = self._arrays()
        keep = scores >= conf
        tp_c = np.bincount(cls[keep & tp], minlength=self.nc)
        fp_c = np.bincount(cls[keep & ~tp], minlength=self.nc)
        fn_c = self.n_gt - tp_c
        per_class = []
        for c in range(self.nc):
            p, r, f1 = self._prf(int(tp_c[c]), int(fp_c[c]), int(fn_c[c]))
            per_class.append(
                {"class_id": c, "precision": p, "recall": r, "f1": f1,
                 "tp": int(tp_c[c]), "fp": int(fp_c[c]), "fn": int(fn_c[c]),
                 "n_gt": int(self.n_gt[c])}
            )
        p, r, f1 = self._prf(int(tp_c.sum()), int(fp_c.sum()), int(fn_c.sum()))
        return {"conf": float(conf), "precision": p, "recall": r, "f1": f1,
                "tp": int(tp_c.sum()), "fp": int(fp_c.sum()), "fn": int(fn_c.sum()),
                "per_class": per_class}

    def average_precision(self) -> np.ndarray:
        """Per-class AP at the evaluator's IoU threshold (101-point interpolation, as in COCO)."""
        scores, cls, tp = self._arrays()
        ap = np.zeros(self.nc)
        recall_points = np.linspace(0, 1, 101)
        for c in range(self.nc):
            if self.n_gt[c] == 0:
                continue
            m = cls == c
            order = np.argsort(-scores[m], kind="stable")
            hits = tp[m][order]
            tpc = np.cumsum(hits)
            fpc = np.cumsum(~hits)
            recall = tpc / self.n_gt[c]
            precision = tpc / np.maximum(tpc + fpc, 1)
            # precision envelope, then sample it at fixed recall points
            envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(precision) else precision
            idx = np.searchsorted(recall, recall_points, side="left")
            ap[c] = np.mean([envelope[i] if i < len(envelope) else 0.0 for i in idx])
        return ap

    def sweep(self, thresholds=DEFAULT_THRESHOLDS) -> list[dict]:
        return [self.at_threshold(t) for t in thresholds]

    def best(self, thresholds=DEFAULT_THRESHOLDS):
        """Returns (best_overall, best_per_class, all_rows)."""
        rows = self.sweep(thresholds)
        best = max(rows, key=lambda r: r["f1"])
        best_pc = []
        for c in range(self.nc):
            br = max(rows, key=lambda r: r["per_class"][c]["f1"])
            best_pc.append({"conf": br["conf"], **br["per_class"][c]})
        return best, best_pc, rows


# --------------------------------------------------------------------------- #
# Results logging
# --------------------------------------------------------------------------- #
def append_metrics_row(row: dict, path=None) -> Path:
    """Append one row to results/metrics.csv (created if missing)."""
    import pandas as pd

    path = Path(path) if path else PROJECT_ROOT / "results" / "metrics.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame([row])
    if path.exists() and path.stat().st_size > 0:
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)
    return path

In [ ]:
%%writefile scripts/fix_prepared_data.py
#!/usr/bin/env python
"""Fix an already-prepared data/{train,val,test}/{images,labels} copy of RDD2022.

Use this instead of convert_voc_to_yolo.py + make_split.py when the dataset was
downloaded pre-converted (e.g. from Kaggle). Two problems are fixed:

1. Class ids. The prepared labels use 5 ids: 0 D00, 1 D10, 2 D20, 3 "other
   corruption" (D43/D44/D50/Repair merged) and 4 D40 pothole. The per-class box
   totals (26016 / 11830 / 10617 / 6544 for ids 0/1/2/4) match the published RDD2022
   counts for D00 / D10 / D20 / D40. The project uses the 4 standard classes, so id 3
   boxes are dropped and id 4 becomes 3. This is only done when id 4 is present.

2. The split. The prepared split is random per image, so near-identical neighbouring
   frames land in train and val/test. Images are regrouped into blocks of consecutive
   file numbers per country (same heuristic as make_split.py) and whole blocks are
   assigned to one split.

In place (default): files are moved between the split folders, so no extra disk.
All original labels and the original split are first saved to
<root>/original_labels.zip. Running twice is refused unless --force is given.

    python scripts/fix_prepared_data.py --root data

To a new folder (--dst): the source is left untouched (e.g. a read-only Kaggle input).
Labels are written to <dst> and images are symlinked (copied if symlinks fail). If the
source was already fixed by this script, its split and classes are mirrored as they are.

    python scripts/fix_prepared_data.py --root /kaggle/input/rdd2022 --dst /kaggle/working/data
"""
from __future__ import annotations

import argparse
import os
import random
import re
import shutil
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

from tqdm import tqdm

from utils import IMG_EXTS, PROJECT_ROOT

SPLITS = ("train", "val", "test")
CLASS_REMAP = {0: 0, 1: 1, 2: 2, 4: 3}  # source id -> project id; anything else is dropped
MARKER = ".fixed_by_fix_prepared_data"


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", default="data", help="folder holding train/, val/, test/")
    p.add_argument("--dst", default=None, help="write the fixed dataset here instead of changing --root")
    p.add_argument("--val-frac", type=float, default=0.15)
    p.add_argument("--test-frac", type=float, default=0.15)
    p.add_argument("--block-size", type=int, default=50, help="consecutive images treated as one sequence")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--keep-classes", action="store_true", help="only re-split, do not remap class ids")
    p.add_argument("--force", action="store_true", help="run even if the marker file says it already ran")
    return p.parse_args()


def country_and_index(stem: str):
    m = re.match(r"(.+?)_(\d+)$", stem)
    return (m.group(1), int(m.group(2))) if m else (stem, None)


def remap_label_text(text: str, dropped: Counter) -> str:
    out = []
    for line in text.splitlines():
        fields = line.split()
        if not fields:
            continue
        cid = int(float(fields[0]))
        if cid not in CLASS_REMAP:
            dropped[cid] += 1
            continue
        out.append(" ".join([str(CLASS_REMAP[cid])] + fields[1:]))
    return "\n".join(out) + ("\n" if out else "")


def has_class_id(label_dirs, class_id: int) -> bool:
    prefix = f"{class_id} "
    for d in label_dirs:
        for f in d.glob("*.txt"):
            with open(f, encoding="utf-8") as fh:
                if any(line.startswith(prefix) for line in fh):
                    return True
    return False


def link_or_copy(src: Path, dst: Path, state: dict) -> None:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if not state["copy"]:
        try:
            os.symlink(src.resolve(), dst)
            return
        except OSError:
            state["copy"] = True
            print("Symlinks are not available here; copying images instead.")
    shutil.copy2(src, dst)


def block_split(items, args) -> dict:
    """Map each (country, block) group to a split, stratified by country."""
    groups = defaultdict(list)
    for item in items:
        country, idx = country_and_index(item[0])
        groups[(country, idx // args.block_size if idx is not None else item[0])].append(item)
    rng = random.Random(args.seed)
    by_country = defaultdict(list)
    for key in groups:
        by_country[key[0]].append(key)
    target = {}
    for country, keys in sorted(by_country.items()):
        keys = sorted(keys, key=str)
        rng.shuffle(keys)
        n = len(keys)
        n_test = max(1, round(n * args.test_frac)) if n >= 3 else 0
        n_val = max(1, round(n * args.val_frac)) if n >= 3 else 0
        for i, key in enumerate(keys):
            target[key] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return groups, target


def main():
    args = parse_args()
    root = Path(args.root)
    if not root.is_absolute():
        root = PROJECT_ROOT / root
    dst = Path(args.dst) if args.dst else root
    if not dst.is_absolute():
        dst = PROJECT_ROOT / dst
    in_place = dst.resolve() == root.resolve()
    if (dst / MARKER).exists() and not args.force:
        sys.exit(f"{dst / MARKER} exists: this data was already fixed. Use --force to run again.")

    # ---- inventory ----
    items = []  # (stem, current_split, image_path)
    for s in SPLITS:
        img_dir = root / s / "images"
        if img_dir.is_dir():
            items += [(p.stem, s, p) for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    if not items:
        sys.exit(f"No images found under {root}/<split>/images")
    stems = [i[0] for i in items]
    if len(set(stems)) != len(stems):
        sys.exit("The same image name appears in more than one split; cannot re-split safely.")

    # splits.csv also counts: uploads to Kaggle can drop dotfiles such as the marker
    source_fixed = ((root / MARKER).exists() or (root / "splits.csv").exists()) and not in_place
    if source_fixed:
        print(f"{root} was already fixed by this script: mirroring its split and classes unchanged.")
        remap = False
    elif args.keep_classes:
        remap = False
    else:
        remap = has_class_id([root / s / "labels" for s in SPLITS if (root / s / "labels").is_dir()], 4)
        print("Class id 4 (D40 pothole) found: remapping to the 4 project classes." if remap else
              "No class id 4 found: labels already use the 4 project classes, keeping them.")

    # ---- backup (in place only; --dst never touches the source) ----
    if in_place:
        backup = root / "original_labels.zip"
        if not backup.exists():
            with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as zf:
                zf.writestr("original_split.csv", "stem,split\n" + "".join(f"{st},{sp}\n" for st, sp, _ in items))
                for stem, s, _ in tqdm(items, desc="Backing up labels"):
                    lbl = root / s / "labels" / f"{stem}.txt"
                    if lbl.exists():
                        zf.write(lbl, f"{s}/labels/{stem}.txt")
            print(f"Backed up original labels and split to {backup}")

    # ---- split ----
    if source_fixed:
        groups = {(stem, None): [(stem, s, img)] for stem, s, img in items}
        target = {key: members[0][1] for key, members in groups.items()}
    else:
        groups, target = block_split(items, args)

    for s in SPLITS:
        (dst / s / "images").mkdir(parents=True, exist_ok=True)
        (dst / s / "labels").mkdir(parents=True, exist_ok=True)

    dropped, moved, state = Counter(), 0, {"copy": False}
    class_counts = {s: Counter() for s in SPLITS}
    rows = []
    for key, members in tqdm(sorted(groups.items(), key=lambda kv: str(kv[0])), desc="Writing split"):
        dst_split = target[key]
        for stem, src_split, img in members:
            src_lbl = root / src_split / "labels" / f"{stem}.txt"
            text = src_lbl.read_text(encoding="utf-8") if src_lbl.exists() else ""
            if remap:
                text = remap_label_text(text, dropped)
            (dst / dst_split / "labels" / f"{stem}.txt").write_text(text, encoding="utf-8")
            if in_place:
                if src_split != dst_split:
                    if src_lbl.exists():
                        src_lbl.unlink()
                    os.replace(img, dst / dst_split / "images" / img.name)
                    moved += 1
            else:
                link_or_copy(img, dst / dst_split / "images" / img.name, state)
                moved += src_split != dst_split
            for line in text.splitlines():
                class_counts[dst_split][int(float(line.split()[0]))] += 1
            rows.append(f"{stem},{key[0]},{key[0]}_{key[1]},{dst_split}\n")

    (dst / "splits.csv").write_text("stem,country,group,split\n" + "".join(sorted(rows)), encoding="utf-8")
    (dst / MARKER).write_text(f"block_size={args.block_size} seed={args.seed} remap={remap}\n", encoding="utf-8")

    # ---- summary ----
    print(f"\n{moved} images changed split. Dropped boxes by source class: {dict(dropped)}")
    per_split = Counter(target[k] for k, m in groups.items() for _ in m)
    for s in SPLITS:
        print(f"{s:>5}: {per_split[s]:>6} images | boxes per class {dict(sorted(class_counts[s].items()))}")
    print(f"\nWrote {dst / 'splits.csv'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/train_faster_rcnn.py
#!/usr/bin/env python
"""Train Faster R-CNN (ResNet-50 FPN) on the prepared YOLO-format RDD2022 data.

Examples:
    python scripts/train_faster_rcnn.py --epochs 30 --imgsz 640 --batch 2 --device 0
    python scripts/train_faster_rcnn.py --resume runs/faster_rcnn/<name>/last.pt --max-hours 11

    # evaluate: pick the threshold on val, then report test F1 at that threshold
    python scripts/train_faster_rcnn.py --eval-weights runs/faster_rcnn/<name>/best.pt --split val
    python scripts/train_faster_rcnn.py --eval-weights runs/faster_rcnn/<name>/best.pt --split test --fixed-conf 0.3

The script converts normalized YOLO boxes to torchvision's pixel-space boxes and
uses COCO-pretrained Faster R-CNN weights with a new four-class prediction head.
Each epoch writes a resumable checkpoint to runs/faster_rcnn/<name>/last.pt.
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import random
import time
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50
from torchvision.models.detection import FasterRCNN, FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection.backbone_utils import _resnet_fpn_extractor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops.misc import FrozenBatchNorm2d
from torchvision.transforms.functional import pil_to_tensor
from tqdm import tqdm

from utils import (PROJECT_ROOT, DetectionEvaluator, append_metrics_row, get_split_dirs, list_images, load_names,
                   resolve_data_yaml, save_json)


class RoadDamageDataset(Dataset):
    def __init__(self, image_dir: Path, label_dir: Path, num_classes: int, fraction: float = 1.0):
        self.images = list_images(image_dir)
        if fraction < 1.0:
            self.images = random.Random(0).sample(self.images, max(1, int(len(self.images) * fraction)))
            self.images.sort()
        self.label_dir = label_dir
        self.num_classes = num_classes
        if not self.images:
            raise FileNotFoundError(f"No images found in {image_dir}")
        missing = [p.stem for p in self.images if not (label_dir / f"{p.stem}.txt").is_file()]
        if missing:
            raise FileNotFoundError(f"Missing {len(missing)} label files in {label_dir}; examples: {missing[:5]}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_path = self.images[index]
        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")
            width, height = image.size
            image_tensor = pil_to_tensor(image).float().div_(255)

        boxes, labels = [], []
        label_path = self.label_dir / f"{image_path.stem}.txt"
        with label_path.open(encoding="utf-8") as f:
            for line_number, line in enumerate(f, start=1):
                fields = line.split()
                if len(fields) != 5:
                    raise ValueError(f"{label_path}:{line_number}: expected 5 YOLO fields")
                try:
                    values = [float(value) for value in fields]
                except ValueError as exc:
                    raise ValueError(f"{label_path}:{line_number}: invalid numeric value") from exc
                if not all(math.isfinite(value) for value in values):
                    raise ValueError(f"{label_path}:{line_number}: values must be finite")
                class_value, cx, cy, box_width, box_height = values
                class_id = int(class_value)
                if class_value != class_id or not 0 <= class_id < self.num_classes:
                    raise ValueError(f"{label_path}:{line_number}: invalid class id {class_value}")
                if box_width <= 0 or box_height <= 0:
                    raise ValueError(f"{label_path}:{line_number}: box width and height must be positive")

                x1 = max(0.0, (cx - box_width / 2) * width)
                y1 = max(0.0, (cy - box_height / 2) * height)
                x2 = min(float(width), (cx + box_width / 2) * width)
                y2 = min(float(height), (cy + box_height / 2) * height)
                if x2 - x1 < 1 or y2 - y1 < 1:
                    continue  # degenerate after clipping; torchvision rejects zero-size boxes
                boxes.append([x1, y1, x2, y2])
                labels.append(class_id + 1)  # torchvision reserves label 0 for background

        target = {
            "size": torch.tensor([height, width], dtype=torch.int64),
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([index], dtype=torch.int64),
        }
        target["area"] = ((target["boxes"][:, 2] - target["boxes"][:, 0]) *
                          (target["boxes"][:, 3] - target["boxes"][:, 1]))
        target["iscrowd"] = torch.zeros((len(boxes),), dtype=torch.int64)
        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def build_model(num_classes: int, imgsz: int, pretrained: bool):
    """Faster R-CNN with the same architecture whether or not COCO weights are loaded.

    With COCO weights torchvision freezes BatchNorm and the first two backbone stages;
    without weights it would build plain BatchNorm instead, which cannot load our
    checkpoints. The no-weights branch rebuilds the pretrained layout explicitly.
    """
    if pretrained:
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
                                         min_size=imgsz, max_size=imgsz)
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
        return model
    backbone = _resnet_fpn_extractor(resnet50(weights=None, norm_layer=FrozenBatchNorm2d), trainable_layers=3)
    return FasterRCNN(backbone, num_classes=num_classes + 1, min_size=imgsz, max_size=imgsz)


@torch.no_grad()
def evaluate(model, loader, device, num_classes, amp=False, return_evaluator=False):
    """F1 @ IoU 0.5 (best confidence threshold) and mAP@0.5, using the same matching as evaluate.py."""
    model.eval()
    evaluator = DetectionEvaluator(num_classes, iou_thr=0.5)
    for images, targets in tqdm(loader, desc="Validating", leave=False):
        images = [image.to(device, non_blocking=device.type == "cuda") for image in images]
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            outputs = model(images)
        for output, target in zip(outputs, targets):
            height, width = target["size"].tolist()
            scale = torch.tensor([width, height, width, height], dtype=torch.float32)
            evaluator.add((output["boxes"].float().cpu() / scale).numpy(), output["scores"].float().cpu().numpy(),
                          (output["labels"].cpu() - 1).numpy(), (target["boxes"] / scale).numpy(),
                          (target["labels"] - 1).numpy())
    best, _, _ = evaluator.best()
    ap50 = evaluator.average_precision()
    metrics = {"precision": best["precision"], "recall": best["recall"], "f1": best["f1"],
               "f1_conf": best["conf"], "map50": float(ap50.mean()), "ap50": ap50.tolist()}
    return (metrics, evaluator) if return_evaluator else metrics


def run_evaluation(args, device, data_yaml, names):
    """Evaluate saved weights on one split and log them like scripts/evaluate.py does."""
    weights = Path(args.eval_weights)
    if not weights.is_absolute():
        weights = PROJECT_ROOT / weights
    checkpoint = torch.load(weights, map_location="cpu", weights_only=True)
    num_classes = len(names)
    if checkpoint["num_classes"] != num_classes:
        raise ValueError(f"{weights} has {checkpoint['num_classes']} classes, the dataset has {num_classes}")
    imgsz = checkpoint.get("imgsz", args.imgsz)
    model = build_model(num_classes, imgsz, pretrained=False)
    model.load_state_dict(checkpoint["model"])
    model.to(device)
    images, labels = get_split_dirs(data_yaml, args.split)
    loader = DataLoader(RoadDamageDataset(images, labels, num_classes, args.fraction), batch_size=1,
                        shuffle=False, num_workers=args.workers, collate_fn=collate_fn)
    start = time.time()
    metrics, evaluator = evaluate(model, loader, device, num_classes, args.amp, return_evaluator=True)
    ms_per_img = 1000 * (time.time() - start) / len(loader.dataset)
    fixed = evaluator.at_threshold(args.fixed_conf) if args.fixed_conf is not None else None
    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    name = args.name or weights.parent.name
    print(f"\nModel: {name} | split: {args.split} | images: {len(loader.dataset)} | imgsz: {imgsz}")
    print(f"mAP@0.5 = {metrics['map50']:.4f}")
    print(f"Best F1@IoU0.5 = {metrics['f1']:.4f} at conf {metrics['f1_conf']:.2f} "
          f"(P {metrics['precision']:.3f}, R {metrics['recall']:.3f})")
    if fixed:
        print(f"F1 at fixed conf {fixed['conf']:.2f} = {fixed['f1']:.4f} "
              f"(P {fixed['precision']:.3f}, R {fixed['recall']:.3f})")
    print(f"Speed (incl. data loading): {ms_per_img:.1f} ms/img | params: {params_m:.1f}M")
    for c, n in names.items():
        print(f"  {n}: AP50 {metrics['ap50'][c]:.3f}")

    json_path = PROJECT_ROOT / "results" / "evaluations" / f"{name}_{args.split}.json"
    save_json({"name": name, "weights": str(weights), "split": args.split, "imgsz": imgsz, "metrics": metrics,
               "fixed_conf": ({k: v for k, v in fixed.items() if k != "per_class"} if fixed else None),
               "ms_per_img": ms_per_img, "params_m": params_m}, json_path)
    row = {"run": name, "split": args.split, "imgsz": imgsz, "params_M": params_m,
           "precision": metrics["precision"], "recall": metrics["recall"], "f1": metrics["f1"],
           "f1_conf": metrics["f1_conf"], "map50": metrics["map50"], "map50_95": None,
           "ms_per_img": ms_per_img, "fps": 1000 / ms_per_img}
    if fixed:
        row.update({"fixed_conf": fixed["conf"], "f1_at_fixed_conf": fixed["f1"]})
    for c, n in names.items():
        row[f"ap50_{n}"] = metrics["ap50"][c]
    print(f"Saved {json_path}\nAppended a row to {append_metrics_row(row)}")


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", default="configs/rdd2022.yaml")
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch", type=int, default=1, help="Use a small batch on GPUs with limited memory")
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default=None, help="e.g. 0, cuda:0 or cpu (default: auto)")
    parser.add_argument("--lr", type=float, default=None,
                        help="base learning rate (default: 0.02 * batch / 16, the torchvision reference scaling)")
    parser.add_argument("--warmup-iters", type=int, default=1000)
    parser.add_argument("--val-every", type=int, default=1, help="validate every N epochs (0 = never)")
    parser.add_argument("--fraction", type=float, default=1.0, help="train/val on this fraction of images")
    parser.add_argument("--patience", type=int, default=30, help="stop after N validations without mAP gain")
    parser.add_argument("--momentum", type=float, default=0.9)
    parser.add_argument("--weight-decay", type=float, default=0.0005)
    parser.add_argument("--amp", action="store_true", help="use CUDA automatic mixed precision")
    parser.add_argument("--project", default="runs/faster_rcnn")
    parser.add_argument("--name", default=None)
    parser.add_argument("--resume", default=None, help="resume from a checkpoint written by this script")
    parser.add_argument("--max-hours", type=float, default=None,
                        help="stop after the epoch that would overrun this budget; continue with --resume last.pt")
    parser.add_argument("--eval-weights", default=None, help="only evaluate these weights (best.pt / model.pt)")
    parser.add_argument("--split", default="val", help="split for --eval-weights: val or test")
    parser.add_argument("--fixed-conf", type=float, default=None, help="with --eval-weights: also report F1 here")
    return parser.parse_args()


def save_checkpoint(payload, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def main():
    args = parse_args()
    if args.imgsz <= 0 or args.epochs <= 0 or args.batch <= 0 or args.workers < 0:
        raise SystemExit("--imgsz, --epochs and --batch must be positive; --workers cannot be negative")
    if args.lr is None:
        args.lr = 0.02 * args.batch / 16
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    requested_device = args.device
    if requested_device and requested_device.isdigit():
        requested_device = f"cuda:{requested_device}"
    if requested_device:
        device = torch.device(requested_device)
    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda" and not torch.cuda.is_available():
        raise SystemExit("CUDA was requested but is unavailable in this PyTorch environment.")

    data_yaml = resolve_data_yaml(args.data)
    names = load_names(data_yaml)
    num_classes = len(names)
    if args.eval_weights:
        run_evaluation(args, device, data_yaml, names)
        return
    train_images, train_labels = get_split_dirs(data_yaml, "train")
    dataset = RoadDamageDataset(train_images, train_labels, num_classes, args.fraction)
    loader = DataLoader(dataset, batch_size=args.batch, shuffle=True, num_workers=args.workers,
                        pin_memory=device.type == "cuda", collate_fn=collate_fn,
                        persistent_workers=args.workers > 0)
    val_loader = None
    if args.val_every > 0:
        val_images, val_labels = get_split_dirs(data_yaml, "val")
        val_dataset = RoadDamageDataset(val_images, val_labels, num_classes, args.fraction)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                                num_workers=args.workers, collate_fn=collate_fn,
                                persistent_workers=args.workers > 0)

    if args.resume:
        resume_path = Path(args.resume)
        if not resume_path.is_absolute():
            resume_path = PROJECT_ROOT / resume_path
        if not resume_path.is_file():
            raise FileNotFoundError(f"Checkpoint not found: {resume_path}")
        checkpoint = torch.load(resume_path, map_location="cpu", weights_only=True)
        model = build_model(num_classes, args.imgsz, pretrained=False)
    else:
        checkpoint = None
        model = build_model(num_classes, args.imgsz, pretrained=True)

    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum,
                                weight_decay=args.weight_decay)
    # torchvision reference schedule: x0.1 at ~2/3 and ~8/9 of training, plus a short linear warmup
    milestones = [max(1, round(args.epochs * 2 / 3)), max(1, round(args.epochs * 8 / 9))]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=0.1)

    run_name = args.name or f"fasterrcnn_resnet50fpn_{args.imgsz}_s{args.seed}"
    save_dir = (PROJECT_ROOT / args.project / run_name).resolve()
    save_dir.mkdir(parents=True, exist_ok=True)
    settings = vars(args).copy()
    settings.update({"device": str(device), "class_names": names, "train_images": len(dataset)})
    with (save_dir / "cli_args.json").open("w", encoding="utf-8") as f:
        json.dump(settings, f, indent=2)

    start_epoch = 0
    if checkpoint is not None:
        if checkpoint["num_classes"] != num_classes or checkpoint["imgsz"] != args.imgsz:
            raise ValueError("Checkpoint class count or image size does not match this run.")
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = int(checkpoint["epoch"])
    best_map50 = float(checkpoint.get("best_map50", -1.0)) if checkpoint else -1.0
    stale = int(checkpoint.get("stale", 0)) if checkpoint else 0

    history_path = save_dir / "metrics.csv"
    if not history_path.exists():
        with history_path.open("w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["epoch", "train_loss", "learning_rate", "precision", "recall", "f1",
                                    "f1_conf", "map50"] + [f"ap50_{n}" for n in names.values()])

    scaler = torch.amp.GradScaler("cuda", enabled=args.amp and device.type == "cuda")
    warmup_iters = min(args.warmup_iters, len(loader)) if start_epoch == 0 else 0
    print(f"Training Faster R-CNN on {device}: {len(dataset)} images, {num_classes} classes, "
          f"batch={args.batch}, epochs={args.epochs}")

    session_start = time.time()
    for epoch in range(start_epoch, args.epochs):
        epoch_start = time.time()
        model.train()
        loss_total = 0.0
        for step, (images, targets) in enumerate(tqdm(loader, desc=f"Epoch {epoch + 1}/{args.epochs}")):
            if epoch == start_epoch and step < warmup_iters:
                warmup = 0.001 + (1 - 0.001) * step / warmup_iters
                for group in optimizer.param_groups:
                    group["lr"] = args.lr * warmup
            elif epoch == start_epoch and warmup_iters and step == warmup_iters:  # never on resume
                for group in optimizer.param_groups:
                    group["lr"] = args.lr
            images = [image.to(device, non_blocking=device.type == "cuda") for image in images]
            targets = [{key: value.to(device, non_blocking=device.type == "cuda") for key, value in target.items()
                        if key != "size"} for target in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=args.amp and device.type == "cuda"):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}: {loss.item()}")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_total += loss.detach().item()

        learning_rate = optimizer.param_groups[0]["lr"]
        scheduler.step()
        mean_loss = loss_total / len(loader)

        metrics = None
        if val_loader is not None and ((epoch + 1) % args.val_every == 0 or epoch + 1 == args.epochs):
            del images, targets, loss_dict, loss  # free training tensors so eval fits on a 4 GB GPU
            if device.type == "cuda":
                torch.cuda.empty_cache()
            metrics = evaluate(model, val_loader, device, num_classes, args.amp)
            if device.type == "cuda":
                torch.cuda.empty_cache()
            if metrics["map50"] > best_map50:
                best_map50, stale = metrics["map50"], 0
                save_checkpoint({"model": model.state_dict(), "num_classes": num_classes, "imgsz": args.imgsz,
                                 "class_names": names, "epoch": epoch + 1, "val": metrics}, save_dir / "best.pt")
            else:
                stale += 1
        row = [epoch + 1, mean_loss, learning_rate]
        if metrics:
            row += [metrics[k] for k in ("precision", "recall", "f1", "f1_conf", "map50")] + metrics["ap50"]
        with history_path.open("a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(row)
        checkpoint_data = {"epoch": epoch + 1, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                           "scheduler": scheduler.state_dict(), "num_classes": num_classes, "imgsz": args.imgsz,
                           "class_names": names, "train_loss": mean_loss, "best_map50": best_map50, "stale": stale}
        save_checkpoint(checkpoint_data, save_dir / "last.pt")
        message = f"Epoch {epoch + 1}/{args.epochs}: train_loss={mean_loss:.4f}, lr={learning_rate:.6g}"
        if metrics:
            message += (f", val P={metrics['precision']:.3f} R={metrics['recall']:.3f} F1={metrics['f1']:.3f}"
                        f" mAP50={metrics['map50']:.4f} (best {best_map50:.4f})")
        print(message, flush=True)
        if val_loader is not None and stale >= args.patience:
            print(f"Early stopping: no mAP@0.5 improvement in {args.patience} validations.")
            break
        epoch_seconds = time.time() - epoch_start
        if (args.max_hours and epoch + 1 < args.epochs
                and time.time() - session_start + 1.2 * epoch_seconds > args.max_hours * 3600):
            (save_dir / "PAUSED").write_text(f"stopped after epoch {epoch + 1}\n", encoding="utf-8")
            print(f"Time budget of {args.max_hours} h reached after epoch {epoch + 1}/{args.epochs}. "
                  f"Continue with: --resume {save_dir / 'last.pt'}")
            return

    (save_dir / "PAUSED").unlink(missing_ok=True)
    save_checkpoint({"model": model.state_dict(), "num_classes": num_classes, "imgsz": args.imgsz,
                     "class_names": names}, save_dir / "model.pt")
    print(f"Training complete. Final weights: {save_dir / 'model.pt'}")
    if best_map50 >= 0:
        print(f"Best weights (val mAP@0.5 = {best_map50:.4f}): {save_dir / 'best.pt'}")
    print(f"Resumable checkpoint: {save_dir / 'last.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/rdd2022.yaml
# Prepared layout data/{train,val,test}/{images,labels}, fixed with scripts/fix_prepared_data.py
# (classes remapped to the 4 below, split by 50-frame blocks per country).
path: data/train
train: images
val: ../val/images
test: ../test/images

names:
  0: D00   # longitudinal crack
  1: D10   # transverse crack
  2: D20   # alligator crack
  3: D40   # pothole

## 4. Data

Finds the dataset, then writes a fixed copy (4 classes, leak-free block split) to the working folder. Images are symlinked, so this is fast and small.

In [ ]:
def find_dataset(roots, max_depth=5):
    """First folder below the roots that has train/images and val/images."""
    skip = {"images", "labels", "drive", "rdd", "outputs", "sample_data", ".config"}
    for root in map(Path, roots):
        if not root.is_dir():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            here = Path(dirpath)
            if (here / "train" / "images").is_dir() and (here / "val" / "images").is_dir():
                return here
            if len(here.relative_to(root).parts) >= max_depth:
                dirnames[:] = []
            else:
                dirnames[:] = [d for d in dirnames if d not in skip]
    return None


src = Path(DATA_DIR) if DATA_DIR else None
if src and src.suffix == ".zip":
    raw = WORK / "raw"
    if not raw.exists():
        print(f"Unzipping {src} ...")
        shutil.unpack_archive(str(src), str(raw))
    src = find_dataset([raw])
elif src is None:
    src = find_dataset(["/kaggle/input"] if ON_KAGGLE else ["/content"] if ON_COLAB else [Path.cwd()])
elif not (src / "train" / "images").is_dir():
    src = find_dataset([src])
if src is None:
    raise FileNotFoundError("Dataset not found. Add it as an input (Kaggle) or set DATA_DIR to the folder or .zip "
                            "that contains train/ val/ test/ with images/ and labels/.")
print("Dataset source:", src)

DATA = WORK / "data"
if not (DATA / ".fixed_by_fix_prepared_data").exists():
    run([sys.executable, "scripts/fix_prepared_data.py", "--root", src, "--dst", DATA])
else:
    print("Prepared data already in", DATA)

## 5. Train

Resumes automatically if a resumable checkpoint exists. Stops cleanly after `MAX_HOURS`; run the notebook again to continue.

In [ ]:
def resumable(pt):
    """True if the checkpoint still has its optimizer state (training can continue from it)."""
    if not Path(pt).is_file():
        return False
    ckpt = torch.load(pt, map_location="cpu", weights_only=False)
    return ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) != -1


def bring_back_previous_run(run_dir):
    """Copy a previous session's run folder (from RESUME_FROM) into place, once."""
    if not RESUME_FROM or run_dir.exists():
        return
    matches = [Path(p) for p in glob.glob(f"{RESUME_FROM}/**/{RUN_NAME}", recursive=True) if Path(p).is_dir()]
    if Path(RESUME_FROM).name == RUN_NAME:
        matches.insert(0, Path(RESUME_FROM))
    if not matches:
        raise FileNotFoundError(f"No folder named {RUN_NAME} under RESUME_FROM={RESUME_FROM}")
    shutil.copytree(matches[0], run_dir)
    print(f"Copied previous run {matches[0]} -> {run_dir}")

In [ ]:
run_dir = RUNS / "faster_rcnn" / RUN_NAME
bring_back_previous_run(run_dir)

if (run_dir / "model.pt").exists() and not (run_dir / "PAUSED").exists():
    print("Training already finished:", run_dir)
else:
    cmd = [sys.executable, "scripts/train_faster_rcnn.py", "--imgsz", IMGSZ, "--epochs", EPOCHS, "--batch", BATCH,
           "--seed", SEED, "--device", DEVICE.split(",")[0], "--workers", min(WORKERS, 4),
           "--project", RUNS / "faster_rcnn", "--name", RUN_NAME] + shlex.split(EXTRA_ARGS)
    if AMP and DEVICE != "cpu":
        cmd.append("--amp")
    if resumable(run_dir / "last.pt"):
        print("Resuming from", run_dir / "last.pt")
        cmd += ["--resume", run_dir / "last.pt"]
    if MAX_HOURS:
        cmd += ["--max-hours", MAX_HOURS]
    run(cmd)

FINISHED = (run_dir / "model.pt").exists() and not (run_dir / "PAUSED").exists()
print("Finished" if FINISHED else "PAUSED: run this notebook again (with RESUME_FROM on Kaggle) to continue.")

## 6. Training curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(run_dir / "metrics.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.plot(x="epoch", y="train_loss", ax=axes[0], title="Training loss")
df.plot(x="epoch", y=["map50", "f1", "precision", "recall"], ax=axes[1], title="Validation metrics")
plt.tight_layout()
plt.show()
df.tail()

## 7. Evaluate on val and test

F1 @ IoU 0.5 (the RDD challenge metric) and mAP@0.5, overall and per class.

In [ ]:
# Pick the confidence threshold on val, then report test F1 at exactly that threshold (honest numbers).
if FINISHED:
    best = run_dir / "best.pt"
    base = [sys.executable, "scripts/train_faster_rcnn.py", "--eval-weights", best, "--imgsz", IMGSZ,
            "--device", DEVICE.split(",")[0], "--workers", min(WORKERS, 4), "--name", RUN_NAME]
    run(base + ["--split", "val"])
    val = json.loads((WORK / "results" / "evaluations" / f"{RUN_NAME}_val.json").read_text())
    run(base + ["--split", "test", "--fixed-conf", val["metrics"]["f1_conf"]])
else:
    print("Training not finished yet; skipping evaluation.")

## 8. Save outputs

In [ ]:
# Copy evaluation results next to the runs and zip the run folder for easy download.
import pandas as pd

if (WORK / "results").exists():
    shutil.copytree(WORK / "results", OUT / "results", dirs_exist_ok=True)
archive = shutil.make_archive(str(OUT / RUN_NAME), "zip", run_dir)
print("Run folder:", run_dir)
print("Zipped    :", archive)
metrics_csv = OUT / "results" / "metrics.csv"
if metrics_csv.exists():
    cols = ["run", "split", "precision", "recall", "f1", "f1_conf", "fixed_conf", "f1_at_fixed_conf",
            "map50", "map50_95", "fps"]
    table = pd.read_csv(metrics_csv)
    display(table[[c for c in cols if c in table.columns]])